# **Data Cleaning Using Python module pandas**

In [2]:
# Import Required module 
import pyarrow 
import os 
import pandas as pd
import logging 
from tabulate import tabulate 


# Logger initializeation
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)


#AWS authentication configuration
try: 
    os.environ["AWS_ACCESS_KEY"] = os.getenv("AWS_ACCESS_KEY")
    os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")
    os.environ["AWS_DEFAULT_REGION"] = os.getenv("AWS_REGION")

    logger.info("AWS authentication configuration loaded successfully.")

except Exception as error:
    logger.exception(
        "Error occurred while loading AWS authentication configuration."
    )
    raise


# fetched data from s3 bucket 
try:
    df = pd.read_csv("s3://ritsky-bucket-2025/retail_raw_dataset/raw_customers.csv")
    logger.info("Data fetched successfully from the S3 bucket.")

except Exception as error:
    logger.exception("Failed to fetch data from the S3 bucket.")


# count the total row and column of source dataset 
logger.info(
    f"Source dataset contains {df.shape[0]} records and {df.shape[1]} columns."
)


# breaking dataset using domain logic
customer_identity = ['customer_id', 'title', 'first_name', 'last_name', 'gender', 'is_active']
customer_address = ['customer_id', 'address', 'city', 'state', 'state_abbr', 'zip_code', 'country', 'region']
customer_content = ['customer_id', 'email', 'phone']
customer_business_info = ['customer_id','customer_segment', 'loyalty_points','preferred_channel', 'annual_income_usd', 'company' ]
customer_dates = ['customer_id','date_of_birth', 'account_created_date']


# Creating DataFrame for eatch domain 
try :
    customer_identity_df = df[customer_identity]
    customer_address_df = df[customer_address]
    customer_content_df = df[customer_content]
    customer_business_info_df = df[customer_business_info]
    customer_dates_df = df[customer_dates]

    logger.info("Customer domain DataFrames created successfully.")

    logger.info(f"customer_identity_df domain contains {customer_identity_df.shape[0]} record and {customer_identity_df.shape[1]} columns.")
    logger.info(f"customer_address_df domain contains {customer_address_df.shape[0]} record and {customer_address_df.shape[1]} columns.")
    logger.info(f"customer_content_df domain contains {customer_content_df.shape[0]} record and {customer_content_df.shape[1]} columns.")
    logger.info(f"customer_business_info_df domain contains {customer_business_info_df.shape[0]} record and {customer_business_info_df.shape[1]} columns.")
    logger.info(f"customer_dates_df domain contains {customer_dates_df.shape[0]} record and {customer_dates_df.shape[1]} columns.")

except Exception as error : 
    logger.exception("Failed to create customer domain DataFrames.")
    raise


required_columns = (
    customer_identity
    + customer_address
    + customer_content
    + customer_business_info
    + customer_dates
)

missing_columns = set(required_columns) - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )
else : 
    logger.info("All DataFrame column match to required_columns.")

2026-07-07 13:39:27,423 | INFO | AWS authentication configuration loaded successfully.
2026-07-07 13:39:29,641 | INFO | Found credentials in shared credentials file: ~/.aws/credentials
2026-07-07 13:39:36,043 | INFO | Data fetched successfully from the S3 bucket.
2026-07-07 13:39:36,046 | INFO | Source dataset contains 640 records and 25 columns.
2026-07-07 13:39:36,073 | INFO | Customer domain DataFrames created successfully.
2026-07-07 13:39:36,076 | INFO | customer_identity_df domain contains 640 record and 6 columns.
2026-07-07 13:39:36,078 | INFO | customer_address_df domain contains 640 record and 8 columns.
2026-07-07 13:39:36,080 | INFO | customer_content_df domain contains 640 record and 3 columns.
2026-07-07 13:39:36,082 | INFO | customer_business_info_df domain contains 640 record and 6 columns.
2026-07-07 13:39:36,084 | INFO | customer_dates_df domain contains 640 record and 3 columns.
2026-07-07 13:39:36,088 | INFO | All DataFrame column match to required_columns.


#### **Handling Missing Values**

In [3]:
# .isnull is the boolean data frame it's return output in form of true and false 
customer_business_info_df.isnull().head()

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
0,False,False,False,False,False,True
1,False,False,False,False,False,False
2,False,False,False,False,False,True
3,False,False,False,False,False,True
4,False,False,False,False,False,True


In [30]:
# .isna is the alias of .isnull both doing same work 
customer_business_info_df.isna().head()

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
0,False,False,False,False,False,True
1,False,False,False,False,False,False
2,False,False,False,False,False,True
3,False,False,False,False,False,True
4,False,False,False,False,False,True


In [36]:
customer_business_info_df.notna().head()

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
0,True,True,True,True,True,False
1,True,True,True,True,True,True
2,True,True,True,True,True,False
3,True,True,True,True,True,False
4,True,True,True,True,True,False


#### **Drop null values** 

In [37]:
business_info_df = customer_business_info_df

In [43]:
# drop rows with ANY NaN
business_info_df.dropna().head()

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
1,1170,Silver,5841.0,In-Store,57415.0,Horizon Red Corp
6,1138,Platinum,9893.0,in-store,93485.0,Alpha Central Corp
8,1342,Gold,14633.0,ONLINE,207173.0,Alpha Pacific Co
13,1387,Gold,9550.0,Mobile,242653.0,Prime Prime Associates
25,9488,Silver,7775.0,catalog,245018.0,Future Elite Industries


In [45]:
# drop columns with ANY NaN
business_info_df.dropna(axis=1).head()

,customer_id,customer_segment,preferred_channel
0,1099,Silver,ONLINE
1,1170,Silver,In-Store
2,1457,Platinum,Catalog
3,1449,Bronze,IN STORE
4,1081,Bronze,mobile


In [47]:
# drop rows where ALL values are NaN 
business_info_df.dropna(how='all').head()

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
0,1099,Silver,7296.0,ONLINE,245670.0,NaN
1,1170,Silver,5841.0,In-Store,57415.0,Horizon Red Corp
2,1457,Platinum,3458.0,Catalog,234384.0,NaN
3,1449,Bronze,7108.0,IN STORE,231000.0,NaN
4,1081,Bronze,6602.0,mobile,206014.0,NaN


In [49]:
business_info_df.dropna(subset=['loyalty_points', 'company']).head()

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
1,1170,Silver,5841.0,In-Store,57415.0,Horizon Red Corp
6,1138,Platinum,9893.0,in-store,93485.0,Alpha Central Corp
8,1342,Gold,14633.0,ONLINE,207173.0,Alpha Pacific Co
13,1387,Gold,9550.0,Mobile,242653.0,Prime Prime Associates
25,9488,Silver,7775.0,catalog,245018.0,Future Elite Industries


In [51]:
# keep rows with at least 3 non-NaN
business_info_df.dropna(thresh=3).head()

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
0,1099,Silver,7296.0,ONLINE,245670.0,NaN
1,1170,Silver,5841.0,In-Store,57415.0,Horizon Red Corp
2,1457,Platinum,3458.0,Catalog,234384.0,NaN
3,1449,Bronze,7108.0,IN STORE,231000.0,NaN
4,1081,Bronze,6602.0,mobile,206014.0,NaN


#### **Fill null value**

In [53]:
# Fill all null value with 0 
business_info_df.fillna(0).head()

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
0,1099,Silver,7296.0,ONLINE,245670.0,0
1,1170,Silver,5841.0,In-Store,57415.0,Horizon Red Corp
2,1457,Platinum,3458.0,Catalog,234384.0,0
3,1449,Bronze,7108.0,IN STORE,231000.0,0
4,1081,Bronze,6602.0,mobile,206014.0,0


In [55]:
# column-specific filling null value 
business_info_df.fillna(
    {
        'loyalty_points' : 0.0 , 
        'company'        : 'Unknown'
    }
).head(10)

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
0,1099,Silver,7296.0,ONLINE,245670.0,Unknown
1,1170,Silver,5841.0,In-Store,57415.0,Horizon Red Corp
2,1457,Platinum,3458.0,Catalog,234384.0,Unknown
3,1449,Bronze,7108.0,IN STORE,231000.0,Unknown
4,1081,Bronze,6602.0,mobile,206014.0,Unknown
5,1247,Silver,4816.0,Catalog,160952.0,Unknown
6,1138,Platinum,9893.0,in-store,93485.0,Alpha Central Corp
7,1517,Platinum,12737.0,In-Store,149325.0,Unknown
8,1342,Gold,14633.0,ONLINE,207173.0,Alpha Pacific Co
9,1597,Bronze,9899.0,Online,73183.0,Unknown


In [63]:
# relpacing null value in loyalty_point with mean 
business_info_df.fillna(
    {
        'loyalty_points' : business_info_df['loyalty_points'].mean(),
        'company' : 'Unknown'
    }
).head()

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
0,1099,Silver,7296.0,ONLINE,245670.0,Unknown
1,1170,Silver,5841.0,In-Store,57415.0,Horizon Red Corp
2,1457,Platinum,3458.0,Catalog,234384.0,Unknown
3,1449,Bronze,7108.0,IN STORE,231000.0,Unknown
4,1081,Bronze,6602.0,mobile,206014.0,Unknown


In [64]:
# relpacing null value in loyalty_point with median 
business_info_df.fillna(
    {
        'loyalty_points' : business_info_df['loyalty_points'].median(),
        'company' : 'Unknown'
    }
).head()

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
0,1099,Silver,7296.0,ONLINE,245670.0,Unknown
1,1170,Silver,5841.0,In-Store,57415.0,Horizon Red Corp
2,1457,Platinum,3458.0,Catalog,234384.0,Unknown
3,1449,Bronze,7108.0,IN STORE,231000.0,Unknown
4,1081,Bronze,6602.0,mobile,206014.0,Unknown


In [67]:
# forward fill rows 
business_info_df['company'].fillna(method='ffill').head()

/tmp/ipykernel_8224/2482260931.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  business_info_df['company'].fillna(method='ffill').head()


0                 NaN
1    Horizon Red Corp
2    Horizon Red Corp
3    Horizon Red Corp
4    Horizon Red Corp
Name: company, dtype: object

In [70]:
# forward full using pandas method ffill
business_info_df['company'].ffill().sample(10)

111        Gold Prime Associates
356           Pacific Pacific Co
484    Horizon National Holdings
74             Global United Inc
131       Metro Prime Associates
116         Summit Gold Holdings
129       Metro Prime Associates
392         Metro Red Associates
196             Summit Metro Inc
151        Apex Pacific Partners
Name: company, dtype: object

In [72]:
# max 2 consecutive fills
business_info_df.fillna(method='pad', limit=2).head()

/tmp/ipykernel_8224/1108205805.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  business_info_df.fillna(method='pad', limit=2).head()


,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
0,1099,Silver,7296.0,ONLINE,245670.0,NaN
1,1170,Silver,5841.0,In-Store,57415.0,Horizon Red Corp
2,1457,Platinum,3458.0,Catalog,234384.0,Horizon Red Corp
3,1449,Bronze,7108.0,IN STORE,231000.0,Horizon Red Corp
4,1081,Bronze,6602.0,mobile,206014.0,NaN


In [74]:
# backward fill
business_info_df['company'].bfill().sample(10)

400               Smart Metro Inc
128    Atlantic Allied Associates
382           Horizon Horizon Ltd
134     United Central Associates
349              Metro Future LLC
444            Next Blue Services
611        Future Future Partners
505         Central Tech Holdings
632     Global Premier Associates
351            Pacific Pacific Co
Name: company, dtype: object